# House Prices: BaseMLP validation-change main baseline

## tl;dr

- 이 노트북은 `validation_test` 산출물이나 외부 프로젝트 스크립트 없이 독립 실행된다.
- BaseMLP 레시피는 유지하고 validation만 5-Fold × 3 seeds로 반복한다.
- 15-fold CV와 3-seed leakage-free OOF를 계산하고 15개 test log 예측을 평균해 submission을 만든다.
- 생성 submission의 SHA-256을 이전 `.12313` repeated-CV 제출과 비교해 완전 동일성을 검증한다.
- W&B에는 15개 fold를 한 group으로 묶고 epoch별 loss·RMSLE·learning rate를 기록한다.

## Context & Methods

이 노트북은 validation-change 실험을 메인 baseline으로 승격한 실행·제출용 artifact다. 데이터 로드, 전처리, `torch.nn.Module`, 학습/검증 loop, early stopping, best checkpoint 복원, OOF, test ensemble, 제출 검사를 모두 포함한다.

### Key Assumptions

- 입력은 `data/train.csv`, `data/test.csv`, `data/sample_submission.csv`다.
- 모든 train 행을 유지하고 역사적 BaseMLP의 `Id=524, YearRemodAdd=2007` modeling-copy 보정을 그대로 둔다.
- 생성 feature는 없고, validation과 ensemble만 5-Fold × seeds `(42, 2026, 3407)`로 고정한다.
- 이전 public score `0.12313`은 선택 입력이 아니라, 동일한 CSV에 대한 과거 확인 결과다.
- W&B는 fold별 run을 만들며 `WANDB_MODE=disabled`로 비활성화할 수 있다.

## Data

### 1. Setup, paths, model, and training functions

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import wandb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data" / "train.csv").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("Could not locate data/train.csv from the current working directory.")

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = ROOT / "notebooks" / "house_prices_basemlp_validation_change.ipynb"
REPORT_DIR = ROOT / "reports"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_PREFIX = "basemlp_validation_change_wandb"
FOLD_RESULTS_PATH = REPORT_DIR / f"{ARTIFACT_PREFIX}_fold_results.csv"
OOF_RESULTS_PATH = REPORT_DIR / f"{ARTIFACT_PREFIX}_oof.csv"
TEST_PREDICTIONS_PATH = REPORT_DIR / f"{ARTIFACT_PREFIX}_test_predictions.csv"
SUMMARY_PATH = REPORT_DIR / f"{ARTIFACT_PREFIX}_summary.json"
SUBMISSION_PATH = (
    ROOT / "submissions" /
    "submission_BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02.csv"
)
RUN_ID = "BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02"
RERUN_OF = "BASEMLP-20260720-VALIDATION-CHANGE-MAIN-NB-01"
EXPECTED_TRAIN_SHA256 = "1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41"
EXPECTED_TEST_SHA256 = "8fdd3d829d4d986b58f845c9553b225e67dd8383624d90fb6ca1d4bed5798c1e"
EXPECTED_SAMPLE_SHA256 = "f19e99699b925aa3085c8d2ae43c62f15fbcd35e4a953b98025f8392c3ffa807"
EXPECTED_REFERENCE_SUBMISSION_SHA256 = "1a428fbc740608a482a24acd1b01f2b8b2d065640d533c62d7ff28c2f1e8b53d"

SPLIT_SEEDS = (42, 2026, 3407)

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "house-prices")
WANDB_ENTITY = os.getenv("WANDB_ENTITY") or None
WANDB_MODE = os.getenv("WANDB_MODE") or None
WANDB_DIR = ARTIFACT_DIR / RUN_ID

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
    split_seed: int,
    fold_in_seed: int,
) -> tuple[BaseMLP, dict[str, object]]:
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        group=experiment_id,
        job_type="cv-fold",
        name=f"seed-{split_seed}-fold-{fold_in_seed}",
        dir=str(WANDB_DIR),
        mode=WANDB_MODE,
        force=False,
        config={
            "experiment_id": experiment_id,
            "rerun_of": RERUN_OF,
            "split_seed": split_seed,
            "fold": fold_in_seed,
            "split_number": fold,
            "model_seed": seed,
            "n_splits": N_SPLITS,
            "training_rows": len(X_train),
            "validation_rows": len(X_valid),
            "input_dim": X_train.shape[1],
            "hidden_dims": list(HIDDEN_DIMS),
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "target": "fold-standardized log1p(SalePrice)",
        },
    )
    wandb_run.define_metric("epoch")
    wandb_run.define_metric("train/loss", step_metric="epoch", summary="min")
    wandb_run.define_metric("validation/loss", step_metric="epoch", summary="min")
    wandb_run.define_metric("validation/rmsle", step_metric="epoch", summary="min")
    wandb_run.define_metric("optimizer/learning_rate", step_metric="epoch", summary="last")
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_row = {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        epoch_rows.append(epoch_row)
        wandb_run.log({
            "epoch": epoch,
            "train/loss": epoch_row["train_loss"],
            "validation/loss": epoch_row["validation_loss"],
            "validation/rmsle": valid_rmsle,
            "optimizer/learning_rate": epoch_row["learning_rate"],
        })

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    wandb_run.summary.update({
        "best/epoch": best_epoch,
        "best/validation_rmsle": best_score,
        "restored/validation_rmsle": restored_rmsle,
        "stopping_epoch": epoch,
        "checkpoint_path": str(checkpoint_path.relative_to(ROOT)),
        "epoch_log_path": str(epoch_log_path.relative_to(ROOT)),
    })
    wandb_run_id = wandb_run.id
    wandb_run_url = wandb_run.url
    wandb_run_dir = wandb_run.dir
    wandb_run_mode = str(wandb_run.settings.mode)
    wandb_run.finish()
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
        "wandb_run_id": wandb_run_id,
        "wandb_run_url": wandb_run_url or "",
        "wandb_run_dir": wandb_run_dir,
        "wandb_run_mode": wandb_run_mode,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert sha256(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256(TEST_PATH) == EXPECTED_TEST_SHA256
assert sha256(SAMPLE_SUBMISSION_PATH) == EXPECTED_SAMPLE_SHA256
assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"test: {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("external project script imports: 0; validation_test dependencies: 0")

train: 1,460 rows × 81 columns
test: 1,459 rows × 80 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
external project script imports: 0; validation_test dependencies: 0


## Fixed BaseMLP recipe

### 2. Historical preprocessing, with no generated features

In [2]:
def clean_rows_historical_basemlp(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[
        cleaned["Id"].eq(524), "YearRemodAdd"
    ] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned

X = clean_rows_historical_basemlp(
    train.drop(columns="SalePrice"), categorical_columns
)
X_test = clean_rows_historical_basemlp(
    test, categorical_columns
)
assert float(
    X.loc[X["Id"].eq(524), "YearRemodAdd"].iloc[0]
) == 2007
assert list(X_test.columns) == [
    column for column in X.columns
]
print("Historical BaseMLP preprocessing reproduced; the validation/ensemble is the only experiment change.")


def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

Historical BaseMLP preprocessing reproduced; the validation/ensemble is the only experiment change.


## Repeated 5-fold training

### 3. Train 15 fresh models and restore every best checkpoint

In [3]:
OUTPUT_DIR = ARTIFACT_DIR / RUN_ID
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
PREPROCESSOR_DIR = OUTPUT_DIR / "preprocessors"
EPOCH_LOG_DIR = OUTPUT_DIR / "logs"
for directory in (CHECKPOINT_DIR, PREPROCESSOR_DIR, EPOCH_LOG_DIR, WANDB_DIR):
    directory.mkdir(parents=True, exist_ok=True)

oof_log_by_seed = np.full(
    (len(SPLIT_SEEDS), len(train)), np.nan, dtype=np.float64
)
test_log_predictions: list[np.ndarray] = []
test_prediction_labels: list[str] = []
fold_rows: list[dict[str, object]] = []
started = time.perf_counter()

for seed_index, split_seed in enumerate(SPLIT_SEEDS):
    splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed)
    for fold, (train_index, valid_index) in enumerate(
        splitter.split(X), start=1
    ):
        split_number = seed_index * N_SPLITS + fold
        fold_seed = split_seed + fold
        checkpoint_path = (
            CHECKPOINT_DIR / f"seed_{split_seed}_fold_{fold}_best.pt"
        )
        preprocessor_path = (
            PREPROCESSOR_DIR / f"seed_{split_seed}_fold_{fold}_preprocessor.joblib"
        )
        epoch_log_path = (
            EPOCH_LOG_DIR / f"seed_{split_seed}_fold_{fold}_epochs.csv"
        )

        preprocessor = make_preprocessor(list(NUMERIC_COLUMNS))
        X_train = preprocessor.fit_transform(
            X.iloc[train_index]
        ).astype(np.float32)
        X_valid = preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        X_test_fold = preprocessor.transform(X_test).astype(np.float32)
        joblib.dump(preprocessor, preprocessor_path)

        model, training_result = train_fold(
            X_train,
            y_log[train_index],
            X_valid,
            y_log[valid_index],
            seed=fold_seed,
            experiment_id=RUN_ID,
            fold=split_number,
            checkpoint_path=checkpoint_path,
            epoch_log_path=epoch_log_path,
            split_seed=split_seed,
            fold_in_seed=fold,
        )
        valid_log_prediction = predict_log_target(
            model,
            X_valid,
            training_result["target_mean"],
            training_result["target_std"],
        )
        test_log_prediction = predict_log_target(
            model,
            X_test_fold,
            training_result["target_mean"],
            training_result["target_std"],
        )
        oof_log_by_seed[seed_index, valid_index] = valid_log_prediction
        test_log_predictions.append(test_log_prediction)
        test_prediction_labels.append(f"seed_{split_seed}_fold_{fold}")
        fold_rows.append({
            "split_seed": split_seed,
            "fold": fold,
            "split_number": split_number,
            "model_seed": fold_seed,
            "training_rows": len(train_index),
            "validation_rows": len(valid_index),
            "encoded_features": X_train.shape[1],
            "best_epoch": training_result["best_epoch"],
            "stopping_epoch": training_result["stopping_epoch"],
            "rmsle": training_result["restored_validation_rmsle"],
            "checkpoint_path": str(checkpoint_path.relative_to(ROOT)),
            "preprocessor_path": str(preprocessor_path.relative_to(ROOT)),
            "epoch_log_path": str(epoch_log_path.relative_to(ROOT)),
            "wandb_run_id": training_result["wandb_run_id"],
            "wandb_run_url": training_result["wandb_run_url"],
            "wandb_run_dir": training_result["wandb_run_dir"],
            "wandb_run_mode": training_result["wandb_run_mode"],
        })
        print(
            f"[{split_number:02d}/15] split_seed={split_seed}, fold={fold}, "
            f"RMSLE={training_result['restored_validation_rmsle']:.6f}, "
            f"best_epoch={training_result['best_epoch']}"
        )

duration_seconds = time.perf_counter() - started
fold_results = pd.DataFrame(fold_rows)
fold_results.to_csv(FOLD_RESULTS_PATH, index=False)
assert not np.isnan(oof_log_by_seed).any()
assert len(test_log_predictions) == 15
print(f"15-fold training completed in {duration_seconds:.2f} seconds")
print("Standalone main training completed without validation_test dependencies.")

wandb: Tracking run with wandb version 0.28.1


wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /Users/joonha/workspace/House_Prices/artifacts/BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02/wandb/offline-run-20260720_120211-ntipiclq
wandb: View this run in the terminal with `wandb leet`


wandb: WARNING URL not available in offline run


wandb: 
wandb: Run history:
wandb:                   epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
wandb: optimizer/learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:              train/loss █▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         validation/loss █▅▄▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂
wandb:        validation/rmsle █▅▄▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▂▂▂▂▂
wandb: 
wandb: Run summary:
wandb:                best/epoch 12
wandb:     best/validation_rmsle 0.14089
wandb:           checkpoint_path artifacts/BASEMLP-20...
wandb:                     epoch 37
wandb:            epoch_log_path artifacts/BASEMLP-20...
wandb: restored/validation_rmsle 0.14089
wandb:            stopping_epoch 37
wandb: 


wandb: You can sync this run to the cloud by running:
wandb: wandb sync /Users/joonha/workspace/House_Prices/artifacts/BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02/wandb/offline-run-20260720_120211-ntipiclq


...

wandb: WARNING URL not available in offline run


wandb: 
wandb: Run history:
wandb:                   epoch ▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
wandb: optimizer/learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:              train/loss █▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         validation/loss █▆▃▂▂▁▂▁▁▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▃▂▃▃▃▃▂▃
wandb:        validation/rmsle █▆▃▂▂▁▂▁▁▂▂▂▂▂▂▂▂▂▂▃▃▂▂▂▃▃▃▃▃▃▃▃▃
wandb: 
wandb: Run summary:
wandb:                best/epoch 8
wandb:     best/validation_rmsle 0.10909
wandb:           checkpoint_path artifacts/BASEMLP-20...
wandb:                     epoch 33
wandb:            epoch_log_path artifacts/BASEMLP-20...
wandb: restored/validation_rmsle 0.10909
wandb:            stopping_epoch 33
wandb: 


wandb: You can sync this run to the cloud by running:
wandb: wandb sync /Users/joonha/workspace/House_Prices/artifacts/BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02/wandb/offline-run-20260720_120227-fc81opg2


wandb: Find logs at: /Users/joonha/workspace/House_Prices/artifacts/BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02/wandb/offline-run-20260720_120227-fc81opg2/logs


[15/15] split_seed=3407, fold=5, RMSLE=0.109095, best_epoch=8
15-fold training completed in 16.88 seconds
Standalone main training completed without validation_test dependencies.


## Validation results

### 4. Compute fold and leakage-free OOF metrics

In [4]:
fold_scores = fold_results["rmsle"].to_numpy(dtype=np.float64)
cv_mean = float(fold_scores.mean())
cv_std = float(fold_scores.std(ddof=1))
seed_oof_scores = {
    str(seed): float(
        np.sqrt(np.mean((y_log - oof_log_by_seed[index]) ** 2))
    )
    for index, seed in enumerate(SPLIT_SEEDS)
}
repeated_ensemble_oof_log = oof_log_by_seed.mean(axis=0)
repeated_ensemble_oof_rmsle = float(
    np.sqrt(np.mean((y_log - repeated_ensemble_oof_log) ** 2))
)

oof_results = pd.DataFrame({
    "Id": train["Id"].to_numpy(dtype=np.int64),
    "target_log1p": y_log,
})
for index, split_seed in enumerate(SPLIT_SEEDS):
    oof_results[f"seed_{split_seed}_oof_log_prediction"] = (
        oof_log_by_seed[index]
    )
oof_results["repeated_ensemble_oof_log_prediction"] = (
    repeated_ensemble_oof_log
)
oof_results["repeated_ensemble_squared_log_error"] = (
    y_log - repeated_ensemble_oof_log
) ** 2
oof_results.to_csv(OOF_RESULTS_PATH, index=False)

assert np.isclose(cv_mean, 0.14588108497354113, rtol=0, atol=1e-12)
assert np.isclose(cv_std, 0.026966873871957453, rtol=0, atol=1e-12)
display(fold_results)
display(pd.DataFrame({
    "metric": [
        "15-fold mean RMSLE",
        "15-fold RMSLE SD",
        "leakage-free 3-seed OOF ensemble RMSLE",
    ],
    "value": [cv_mean, cv_std, repeated_ensemble_oof_rmsle],
}))
print("Per-seed OOF RMSLE:", seed_oof_scores)

,split_seed,fold,split_number,model_seed,training_rows,validation_rows,encoded_features,best_epoch,stopping_epoch,rmsle,checkpoint_path,preprocessor_path,epoch_log_path,wandb_run_id,wandb_run_url,wandb_run_dir,wandb_run_mode
0,42,1,1,43,1168,292,327,12,37,0.140891,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,ntipiclq,,/Users/joonha/workspace/House_Prices/artifacts...,offline
1,42,2,2,44,1168,292,327,43,68,0.131788,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,cjx5ly8k,,/Users/joonha/workspace/House_Prices/artifacts...,offline
2,42,3,3,45,1168,292,323,5,30,0.222199,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,4r4z9dan,,/Users/joonha/workspace/House_Prices/artifacts...,offline
3,42,4,4,46,1168,292,324,21,46,0.134316,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,ona9tvjr,,/Users/joonha/workspace/House_Prices/artifacts...,offline
4,42,5,5,47,1168,292,328,14,39,0.122514,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,5v5izy54,,/Users/joonha/workspace/House_Prices/artifacts...,offline
5,2026,1,6,2027,1168,292,323,7,32,0.141128,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,lh3u5qdx,,/Users/joonha/workspace/House_Prices/artifacts...,offline
6,2026,2,7,2028,1168,292,326,30,55,0.179038,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,ht79vlii,,/Users/joonha/workspace/House_Prices/artifacts...,offline
7,2026,3,8,2029,1168,292,325,5,30,0.128823,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,hers38n3,,/Users/joonha/workspace/House_Prices/artifacts...,offline
8,2026,4,9,2030,1168,292,327,11,36,0.149235,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,94y8o0xf,,/Users/joonha/workspace/House_Prices/artifacts...,offline
9,2026,5,10,2031,1168,292,328,11,36,0.141151,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,artifacts/BASEMLP-20260720-VALIDATION-CHANGE-W...,ltsdvh7v,,/Users/joonha/workspace/House_Prices/artifacts...,offline


,metric,value
0,15-fold mean RMSLE,0.145881
1,15-fold RMSLE SD,0.026967
2,leakage-free 3-seed OOF ensemble RMSLE,0.134232


Per-seed OOF RMSLE: {'42': 0.15468769439866675, '2026': 0.14883664497152194, '3407': 0.14071047429531103}


## Submission and exact reproduction

### 5. Average 15 test log predictions, validate format, and reproduce the previous CSV

In [5]:
test_prediction_matrix = np.column_stack(test_log_predictions)
mean_test_log_prediction = test_prediction_matrix.mean(axis=1)
sale_price_prediction = np.expm1(mean_test_log_prediction)

test_prediction_results = pd.DataFrame({
    "Id": test["Id"].to_numpy(dtype=np.int64)
})
for column_index, label in enumerate(test_prediction_labels):
    test_prediction_results[f"{label}_log_prediction"] = (
        test_prediction_matrix[:, column_index]
    )
test_prediction_results["mean_log_prediction"] = mean_test_log_prediction
test_prediction_results["SalePrice"] = sale_price_prediction
test_prediction_results.to_csv(TEST_PREDICTIONS_PATH, index=False)

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
assert sample_submission.columns.tolist() == ["Id", "SalePrice"]
assert len(sample_submission) == len(test)
assert sample_submission["Id"].tolist() == test["Id"].tolist()
submission_frame = sample_submission.copy()
submission_frame["SalePrice"] = sale_price_prediction.astype(np.float64)
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_frame.to_csv(SUBMISSION_PATH, index=False)

generated_submission_sha256 = sha256(SUBMISSION_PATH)
exact_reference_reproduction = (
    generated_submission_sha256 == EXPECTED_REFERENCE_SUBMISSION_SHA256
)
assert exact_reference_reproduction

submission_validation = {
    "columns": submission_frame.columns.tolist(),
    "rows": int(len(submission_frame)),
    "dtypes": {
        column: str(dtype) for column, dtype in submission_frame.dtypes.items()
    },
    "id_matches_test_order": bool(
        submission_frame["Id"].tolist() == test["Id"].tolist()
    ),
    "unique_ids": bool(submission_frame["Id"].is_unique),
    "missing_predictions": int(submission_frame["SalePrice"].isna().sum()),
    "nonfinite_predictions": int(
        (~np.isfinite(submission_frame["SalePrice"])).sum()
    ),
    "nonpositive_predictions": int(
        (submission_frame["SalePrice"] <= 0).sum()
    ),
    "prediction_min": float(submission_frame["SalePrice"].min()),
    "prediction_max": float(submission_frame["SalePrice"].max()),
    "fold_prediction_count": int(test_prediction_matrix.shape[1]),
    "checkpoint_count": len(list(CHECKPOINT_DIR.glob("*.pt"))),
    "preprocessor_count": len(list(PREPROCESSOR_DIR.glob("*.joblib"))),
    "epoch_log_count": len(list(EPOCH_LOG_DIR.glob("*.csv"))),
    "sha256": generated_submission_sha256,
    "matches_previous_validation_submission_sha256": exact_reference_reproduction,
}
submission_validation["all_checks_passed"] = bool(
    submission_validation["columns"] == ["Id", "SalePrice"]
    and submission_validation["rows"] == len(test)
    and submission_validation["id_matches_test_order"]
    and submission_validation["unique_ids"]
    and submission_validation["missing_predictions"] == 0
    and submission_validation["nonfinite_predictions"] == 0
    and submission_validation["nonpositive_predictions"] == 0
    and submission_validation["fold_prediction_count"] == 15
    and submission_validation["checkpoint_count"] == 15
    and submission_validation["preprocessor_count"] == 15
    and submission_validation["epoch_log_count"] == 15
    and exact_reference_reproduction
)
assert submission_validation["all_checks_passed"]

run_timestamp = datetime.now(timezone.utc).isoformat()
summary = {
    "run_timestamp_utc": run_timestamp,
    "experiment_id": RUN_ID,
    "rerun_of": RERUN_OF,
    "role": "standalone main validation-change baseline and submission",
    "independence": {
        "external_project_script_imports": 0,
        "validation_test_artifact_dependencies": 0,
        "required_inputs": [
            str(TRAIN_PATH.relative_to(ROOT)),
            str(TEST_PATH.relative_to(ROOT)),
            str(SAMPLE_SUBMISSION_PATH.relative_to(ROOT)),
        ],
    },
    "change_scope": {
        "validation": (
            "KFold(5, shuffle=True) repeated at split seeds 42, 2026, 3407"
        ),
        "training_policy": "keep_all",
        "test_ensemble": (
            "mean of 15 restored-checkpoint log predictions, then expm1"
        ),
        "model_changes": 0,
        "feature_changes": 0,
        "row_policy_changes": 0,
        "historical_id_524_correction_preserved": True,
    },
    "source": {
        "train_sha256": sha256(TRAIN_PATH),
        "test_sha256": sha256(TEST_PATH),
        "sample_submission_sha256": sha256(SAMPLE_SUBMISSION_PATH),
    },
    "validation": {
        "fold_scores": fold_scores.tolist(),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "seed_oof_rmsle": seed_oof_scores,
        "repeated_ensemble_oof_rmsle": repeated_ensemble_oof_rmsle,
    },
    "model": {
        "class": "BaseMLP(nn.Module)",
        "architecture": "input->[128,64]->1",
        "optimizer": "Adam(lr=0.001, weight_decay=0)",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "device": str(DEVICE),
    },
    "results": {
        "duration_seconds": duration_seconds,
        "best_epochs": fold_results["best_epoch"].astype(int).tolist(),
        "stopping_epochs": fold_results["stopping_epoch"].astype(int).tolist(),
    },
    "tracking": {
        "wandb_version": wandb.__version__,
        "wandb_project": WANDB_PROJECT,
        "wandb_entity": WANDB_ENTITY,
        "wandb_requested_mode": WANDB_MODE or "auto",
        "wandb_observed_modes": sorted(
            fold_results["wandb_run_mode"].unique().tolist()
        ),
        "wandb_group": RUN_ID,
        "wandb_run_count": len(fold_results),
        "wandb_run_ids": fold_results["wandb_run_id"].tolist(),
        "wandb_run_urls": [
            url for url in fold_results["wandb_run_url"].tolist() if url
        ],
    },
    "submission_validation": submission_validation,
    "reference_reproduction": {
        "previous_submission_sha256": EXPECTED_REFERENCE_SUBMISSION_SHA256,
        "new_submission_sha256": generated_submission_sha256,
        "exact_file_match": exact_reference_reproduction,
        "previous_confirmed_public_score": 0.12313,
        "new_file_submitted_to_kaggle": False,
    },
    "artifacts": {
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "fold_results": str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        "oof_predictions": str(OOF_RESULTS_PATH.relative_to(ROOT)),
        "test_predictions": str(TEST_PREDICTIONS_PATH.relative_to(ROOT)),
        "submission": str(SUBMISSION_PATH.relative_to(ROOT)),
        "checkpoint_dir": str(CHECKPOINT_DIR.relative_to(ROOT)),
        "preprocessor_dir": str(PREPROCESSOR_DIR.relative_to(ROOT)),
        "epoch_log_dir": str(EPOCH_LOG_DIR.relative_to(ROOT)),
        "wandb_dir": str((WANDB_DIR / "wandb").relative_to(ROOT)),
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

append_experiment({
    "experiment_id": RUN_ID,
    "datetime": run_timestamp,
    "objective": (
        "Add W&B fold/epoch tracking to the standalone validation-change "
        "notebook and exactly reproduce its prior submission"
    ),
    "data_version": (
        f"train sha256={sha256(TRAIN_PATH)}; test sha256={sha256(TEST_PATH)}"
    ),
    "split_strategy": (
        "Repeated KFold: KFold(5, shuffle=True) at split seeds 42|2026|3407"
    ),
    "folds": "5x3",
    "seed": "42|2026|3407; model seed=split_seed+fold",
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": (
        "Historical BaseMLP fold median/scaling/one-hot; Id=524 "
        "YearRemodAdd=2007 modeling-copy correction preserved"
    ),
    "features": "79 original predictors; Id excluded; no generated features",
    "model": "Manual PyTorch BaseMLP",
    "architecture": (
        "input->128(ReLU)->64(ReLU)->1; Kaiming hidden/Xavier output; "
        "no dropout/batchnorm"
    ),
    "optimizer": "Adam(weight_decay=0)",
    "loss": "MSELoss on fold-standardized log1p(SalePrice)",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "best_epoch": json.dumps(
        fold_results["best_epoch"].astype(int).tolist()
    ),
    "hyperparameters": (
        f"rerun_of={RERUN_OF}; keep_all; min_delta=1e-5; test=mean of 15 "
        f"restored-checkpoint log predictions then expm1; float32; CPU; "
        f"W&B project={WANDB_PROJECT}; group={RUN_ID}; one run per fold"
    ),
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "holdout_score": repeated_ensemble_oof_rmsle,
    "checkpoint_path": str(CHECKPOINT_DIR.relative_to(ROOT)),
    "artifact_path": " | ".join([
        str(SUBMISSION_PATH.relative_to(ROOT)),
        str(SUMMARY_PATH.relative_to(ROOT)),
        str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        str(OOF_RESULTS_PATH.relative_to(ROOT)),
        str(TEST_PREDICTIONS_PATH.relative_to(ROOT)),
    ]),
    "result": (
        "W&B-instrumented standalone notebook created an exact-SHA "
        "reproduction of the previous repeated-CV submission"
    ),
    "interpretation": (
        f"15-fold CV={cv_mean:.6f}; three-seed OOF ensemble="
        f"{repeated_ensemble_oof_rmsle:.6f}; submission SHA exact match; "
        f"W&B fold runs={len(fold_results)}"
    ),
    "next_action": (
        "Sync offline W&B runs or use WANDB_MODE=online after login; use this "
        "notebook as the main baseline and record Kaggle submissions separately"
    ),
})

display(submission_frame.head())
display(pd.DataFrame([{
    "new_submission": str(SUBMISSION_PATH.relative_to(ROOT)),
    "sha256": generated_submission_sha256,
    "exact_previous_submission_match": exact_reference_reproduction,
}]))
print(f"Submission: {SUBMISSION_PATH.relative_to(ROOT)}")
print(f"SHA-256: {generated_submission_sha256}")
print("All submission checks and exact-reference reproduction passed.")


,Id,SalePrice
0,1461,121762.643060
1,1462,160192.617219
2,1463,179977.650306
3,1464,195592.028369
4,1465,187427.242140


,new_submission,sha256,exact_previous_submission_match
0,submissions/submission_BASEMLP-20260720-VALIDA...,1a428fbc740608a482a24acd1b01f2b8b2d065640d533c...,True


Submission: submissions/submission_BASEMLP-20260720-VALIDATION-CHANGE-WANDB-NB-02.csv
SHA-256: 1a428fbc740608a482a24acd1b01f2b8b2d065640d533c62d7ff28c2f1e8b53d
All submission checks and exact-reference reproduction passed.


## Takeaways

- 주 feature 비교 지표는 동일한 15 folds의 mean RMSLE다.
- submission은 15개 restored-checkpoint log 예측을 동일 가중 평균한 뒤 `expm1`한다.
- SHA-256 동일성 검사는 이전 validation-change 제출과 행, 순서, 부동소수 예측, CSV 직렬화가 모두 같다는 뜻이다.
- 새 파일은 아직 Kaggle에 재제출하지 않았으며, 이전 `.12313`은 동일 reference CSV에 대해 확인된 값이다.
- W&B fold run 15개는 experiment ID group으로 묶이며, offline 실행은 로그인 후 `wandb sync`할 수 있다.